# Colab Demo: Supply Chain Chatbot with Gemini Explanations

This notebook demonstrates the coordinated supply chain optimization chatbot in Google Colab.

It shows:

1. Installing Gemini dependencies
2. Setting `GEMINI_API_KEY`
3. Loading the project
4. Building a minimal supply chain problem
5. Using `GeminiExplanationProvider`
6. Running the chatbot with `use_llm=True`
7. Demonstrating graceful fallback to rule-based explanations

In [ ]:
!pip install -q google-generativeai

In [ ]:
import os

repo_dir = "ChatbotLP"

if not os.path.isdir(repo_dir):
    # Replace with your real repository URL
    !git clone https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git {repo_dir}

%cd {repo_dir}
!pip install -q -r requirements.txt

In [ ]:
import os

try:
    from google.colab import userdata
    secret_key = userdata.get("GEMINI_API_KEY")
    if secret_key:
        os.environ["GEMINI_API_KEY"] = secret_key
        print("Loaded GEMINI_API_KEY from Colab secrets.")
    else:
        print("No GEMINI_API_KEY found in Colab secrets.")
except Exception:
    print("Colab secrets not available or key not found.")

if "GEMINI_API_KEY" not in os.environ:
    from getpass import getpass
    manual_key = getpass("Enter GEMINI_API_KEY (leave blank to test fallback mode): ").strip()
    if manual_key:
        os.environ["GEMINI_API_KEY"] = manual_key

print("GEMINI_API_KEY set?", "GEMINI_API_KEY" in os.environ)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from src.schema import (
    ProblemState,
    Node,
    Product,
    Supplier,
    Consumer,
    TransportLink,
    Bid,
)
from src.chatbot_engine import run_chatbot_session, IntentRouter
from src.parser import parse_supply_chain_text
from src.response_generator import generate_response
from src.llm_adapter import RuleBasedProvider, LLMProviderRegistry
from src.gemini_explanation_provider import GeminiExplanationProvider

In [ ]:
def create_minimal_supply_chain_problem():
    state = ProblemState(problem_title="Minimal Supply Chain Demo")

    state.add_node(Node(id="N1", name="Supplier Location"))
    state.add_node(Node(id="N2", name="Consumer Location"))

    state.add_product(Product(id="P1", name="Product A"))

    state.add_supplier(
        Supplier(
            id="S1",
            node="N1",
            product="P1",
            capacity=100.0,
        )
    )

    state.add_consumer(
        Consumer(
            id="C1",
            node="N2",
            product="P1",
            capacity=50.0,
        )
    )

    state.add_transport(
        TransportLink(
            id="T1",
            origin="N1",
            destination="N2",
            product="P1",
            capacity=100.0,
        )
    )

    state.add_bid(
        Bid(
            id="B1",
            owner_id="S1",
            owner_type="supplier",
            product_id="P1",
            price=10.0,
            quantity=100.0,
        )
    )

    state.add_bid(
        Bid(
            id="B2",
            owner_id="C1",
            owner_type="consumer",
            product_id="P1",
            price=20.0,
            quantity=50.0,
        )
    )

    return state


state = create_minimal_supply_chain_problem()

print("Problem created:")
print("Nodes:", [n.id for n in state.nodes])
print("Products:", [p.id for p in state.products])
print("Suppliers:", [s.id for s in state.suppliers])
print("Consumers:", [c.id for c in state.consumers])
print("Transport links:", [t.id for t in state.transport_links])
print("Bids:", [b.id for b in state.bids])

In [ ]:
def register_gemini_provider():
    gemini_provider = GeminiExplanationProvider()

    class GeminiLLMProvider(RuleBasedProvider):
        def get_explanation_generator(self):
            return gemini_provider

    registry = LLMProviderRegistry.get_instance()
    registry.set_provider(
        GeminiLLMProvider(
            intent_router=IntentRouter(),
            parse_function=parse_supply_chain_text,
            generate_function=generate_response,
        )
    )
    return gemini_provider


gemini_provider = None

try:
    gemini_provider = register_gemini_provider()
    print("GeminiExplanationProvider initialized and registered.")
except Exception as e:
    print("Could not initialize GeminiExplanationProvider:", str(e))
    print("The notebook will fall back to the rule-based explanation generator.")

In [ ]:
result = run_chatbot_session(
    state=state,
    user_message="Explain the current supply chain problem",
    mode="guided",
    use_llm=True,
)

print("Intent:", result.get("intent"))
print("Success:", result.get("success"))
print("\nResponse:\n")
print(result.get("response"))

In [ ]:
original_key = os.environ.pop("GEMINI_API_KEY", None)
LLMProviderRegistry.get_instance().reset()

print("GEMINI_API_KEY present after removal?", "GEMINI_API_KEY" in os.environ)

fallback_result = run_chatbot_session(
    state=state,
    user_message="Explain the current supply chain problem",
    mode="guided",
    use_llm=True,
)

print("\nFallback response:\n")
print(fallback_result.get("response"))

if original_key is not None:
    os.environ["GEMINI_API_KEY"] = original_key
    print("\nGEMINI_API_KEY restored.")

## Notes

- If `GEMINI_API_KEY` is available, explanations are generated through Gemini.
- If the key is missing or Gemini initialization fails, the system falls back to the built-in rule-based explanation path.
- The optimization and theorem-checking core remain deterministic.